# AI Symptom & Care Guide

This project is a simple AI-driven healthcare application that analyzes user-entered symptoms and provides a general care recommendation.

I developed and coded this project with some assistance from ChatGPT. I used ChatGPT mainly for brainstorming ideas, coding guidance, troubleshooting, and improving parts of the application. I reviewed the code, made decisions about the project structure and features, and worked through the application as I built it.

The application is for educational purposes only and does not replace professional medical advice.

## 1. Project Setup

The first step is to install the libraries needed for the application. Streamlit will be used to create the user interface, and the app will later be opened from Google Colab through a temporary Cloudflare tunnel.


In [1]:
# Install Streamlit for the web application interface.
!pip install streamlit -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 31.4 MB/s eta 0:00:00


## 2. Import Libraries

The application uses Streamlit to create the interface and Python's regular expression library to help clean and process the symptom text entered by the user.

In [2]:
# Import the libraries needed for the project.

import re
import streamlit as st

## 3. Create the Symptom Knowledge Base

The application needs a small knowledge base that connects common symptoms with possible health categories and recommended care levels.

Each entry contains:

- Keywords that the application will search for
- A possible condition or symptom category
- A recommended care level
- A short explanation of why that care level may be appropriate

The four care levels used in this project are Self-Care, Doctor Visit, Urgent, and Emergency.

In [3]:
# Create a small symptom knowledge base.
# Each category contains keywords, a care level, and an explanation.

symptom_database = {
    "General Headache Symptoms": {
        "keywords": [
            "headache",
            "mild headache"
        ],
        "care_level": "Self-Care",
        "explanation": (
            "A mild headache can often be monitored with rest, hydration, "
            "and other basic self-care unless it becomes severe or unusual."
        )
    },

    "Common Cold": {
        "keywords": [
            "cough",
            "runny nose",
            "stuffy nose",
            "sneezing",
            "sore throat"
        ],
        "care_level": "Self-Care",
        "explanation": (
            "These symptoms are commonly associated with mild respiratory "
            "conditions that can often be monitored at home."
        )
    },

    "Flu-Like Symptoms": {
        "keywords": [
            "fever",
            "chills",
            "body aches",
            "fatigue"
        ],
        "care_level": "Doctor Visit",
        "explanation": (
            "Fever combined with body aches, fatigue, or chills may require "
            "medical evaluation, especially if symptoms become severe."
        )
    },

    "Possible Infection": {
        "keywords": [
            "high fever",
            "swelling",
            "pus",
            "infection",
            "severe sore throat"
        ],
        "care_level": "Doctor Visit",
        "explanation": (
            "These symptoms may indicate an infection that could require "
            "evaluation and treatment by a healthcare professional."
        )
    },

    "Severe Abdominal Symptoms": {
        "keywords": [
            "severe stomach pain",
            "severe abdominal pain",
            "persistent vomiting",
            "blood in stool"
        ],
        "care_level": "Urgent",
        "explanation": (
            "Severe abdominal symptoms may require prompt medical attention "
            "to determine their cause."
        )
    },

    "Breathing or Chest Emergency": {
        "keywords": [
            "chest pain",
            "difficulty breathing",
            "shortness of breath",
            "cannot breathe",
            "blue lips"
        ],
        "care_level": "Emergency",
        "explanation": (
            "Chest pain or serious breathing problems can be signs of a "
            "medical emergency and should receive immediate attention."
        )
    },

    "Neurological Emergency": {
        "keywords": [
            "face drooping",
            "slurred speech",
            "sudden weakness",
            "seizure",
            "unconscious"
        ],
        "care_level": "Emergency",
        "explanation": (
            "Sudden neurological symptoms can indicate a serious emergency "
            "and require immediate professional medical care."
        )
    }
}


## 4. Review the Knowledge Base

Before building the analysis function, I can print the available symptom categories to confirm that the knowledge base was created correctly.

In [4]:
# Display the condition categories stored in the knowledge base.

print("Available symptom categories:\n")

for condition in symptom_database:
    print("-", condition)

Available symptom categories:

- General Headache Symptoms
- Common Cold
- Flu-Like Symptoms
- Possible Infection
- Severe Abdominal Symptoms
- Breathing or Chest Emergency
- Neurological Emergency


## 5. Clean the User's Symptom Text

Users may enter symptoms using capital letters, extra spaces, or punctuation. To make symptom matching more consistent, the application first cleans the text before analyzing it.

In [5]:
def clean_text(text):
    """
    Clean the symptom description entered by the user.

    The function:
    1. Converts the text to lowercase.
    2. Removes unnecessary punctuation.
    3. Removes extra spaces.

    Parameters:
        text (str): The original symptom description.

    Returns:
        str: A cleaned version of the text.
    """

    # Convert everything to lowercase so matching is not case-sensitive.
    text = text.lower()

    # Remove punctuation while keeping letters, numbers, and spaces.
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Replace multiple spaces with one space.
    text = re.sub(r"\s+", " ", text).strip()

    return text

## 6. Test the Text Cleaning Function

This test helps confirm that the preprocessing step works correctly before it is used in the full symptom analysis system.

In [6]:
# Test the cleaning function with a sample sentence.

sample_text = "I have a BAD headache, fever, and sore throat!!!"

cleaned_sample = clean_text(sample_text)

print("Original text:")
print(sample_text)

print("\nCleaned text:")
print(cleaned_sample)

Original text:
I have a BAD headache, fever, and sore throat!!!

Cleaned text:
i have a bad headache fever and sore throat


## 7. Natural Language Processing Setup

To analyze symptom descriptions more effectively, the application uses Natural Language Processing (NLP).

The NLTK library is used to:

- Split the user's text into individual words
- Remove common words that do not provide much meaning
- Reduce words to their basic dictionary form using lemmatization

This helps prepare the symptom description before it is compared with the symptom knowledge base.

In [7]:
# Install NLTK, which will be used for basic Natural Language Processing.
!pip install nltk -q

## 8. Import and Download NLTK Resources

NLTK requires several additional resources for tokenization, stop-word removal, and lemmatization. These resources are downloaded before the symptom processing function is created.

In [8]:
# Import the NLTK tools needed for processing symptom descriptions.

import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


# Download the NLTK resources used by the application.
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

## 9. Process the User's Symptoms

The next function prepares the user's symptom description for analysis.

First, the text is converted to lowercase and tokenized into individual words. Common stop words such as "the," "and," and "I" are removed because they usually do not help identify symptoms. Finally, lemmatization reduces words to their basic form so similar words can be handled more consistently.

In [9]:
# Create the lemmatizer and the English stop-word list once
# so they can be reused each time symptoms are analyzed.

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))


def process_symptoms(text):
    """
    Prepare a user's symptom description for analysis.

    Steps:
    1. Convert the text to lowercase.
    2. Split the text into individual tokens.
    3. Remove punctuation and common stop words.
    4. Lemmatize the remaining words.

    Parameters:
        text (str): Symptom description entered by the user.

    Returns:
        list: A list of processed symptom words.
    """

    # Convert text to lowercase so matching is not case-sensitive.
    text = text.lower()

    # Split the sentence into individual words and punctuation.
    tokens = word_tokenize(text)

    # Keep useful words while removing punctuation and common stop words.
    filtered_tokens = [
        word
        for word in tokens
        if word.isalnum() and word not in stop_words
    ]

    # Convert each word to its basic dictionary form.
    processed_tokens = [
        lemmatizer.lemmatize(word)
        for word in filtered_tokens
    ]

    return processed_tokens

## 10. Test the NLP Processing

Before using the function in the symptom analysis system, I tested it with a sample symptom description. This allows me to see how the original sentence changes after tokenization, stop-word removal, and lemmatization.

In [10]:
# Example symptom description used to test NLP preprocessing.

sample_symptoms = "I have a severe headache, fever, and sore throat."

processed_sample = process_symptoms(sample_symptoms)

print("Original description:")
print(sample_symptoms)

print("\nProcessed symptoms:")
print(processed_sample)

Original description:
I have a severe headache, fever, and sore throat.

Processed symptoms:
['severe', 'headache', 'fever', 'sore', 'throat']


## 11. Analyze Symptoms

The analysis function compares the processed user input with the keywords stored in each symptom category.

Instead of requiring an exact sentence match, the application counts how many symptom keywords match each category. The category with the strongest match becomes the main result.

The application also gives priority to emergency symptoms. If a serious symptom such as chest pain or difficulty breathing is detected, the system will not allow a less serious category to override the emergency recommendation.

In [11]:
# Assign a priority to each care level.
care_priority = {
    "Self-Care": 1,
    "Doctor Visit": 2,
    "Urgent": 3,
    "Emergency": 4
}


def analyze_symptoms(symptoms):
    # Compare the user's symptoms with the symptom knowledge base.
    processed_tokens = process_symptoms(symptoms)
    processed_text = " ".join(processed_tokens)

    best_match = None
    highest_score = 0
    highest_priority = 0

    for condition, information in symptom_database.items():
        current_matches = []

        for keyword in information["keywords"]:
            keyword_tokens = process_symptoms(keyword)

            # Match single-word symptoms directly in the processed word list.
            if len(keyword_tokens) == 1:
                if keyword_tokens[0] in processed_tokens:
                    current_matches.append(keyword)

            # Match longer symptom phrases in the processed text.
            else:
                processed_keyword = " ".join(keyword_tokens)
                if processed_keyword in processed_text:
                    current_matches.append(keyword)

        score = len(current_matches)
        priority = care_priority[information["care_level"]]

        if score > highest_score or (
            score == highest_score
            and score > 0
            and priority > highest_priority
        ):
            highest_score = score
            highest_priority = priority

            best_match = {
                "condition": condition,
                "care_level": information["care_level"],
                "explanation": information["explanation"],
                "matched_symptoms": current_matches,
                "score": score
            }

    if best_match is None:
        return {
            "condition": "No Clear Match",
            "care_level": "Doctor Visit",
            "explanation": (
                "The symptoms did not clearly match the categories in this "
                "project. Consider speaking with a healthcare professional "
                "if symptoms continue or become concerning."
            ),
            "matched_symptoms": [],
            "score": 0
        }

    return best_match


## 12. Test the Symptom Analysis

I tested the analysis function with several symptom descriptions representing different levels of care. Testing different examples helps confirm that the application can distinguish between mild symptoms and symptoms that may require urgent or emergency attention.

In [12]:
# Create several sample cases to test the symptom analysis function.

test_cases = [
    "I have a cough, sore throat, and runny nose.",
    "I have a fever, chills, headache, and body aches.",
    "I have severe stomach pain and persistent vomiting.",
    "I have chest pain and difficulty breathing."
]


# Analyze each test case and display the results.
for symptoms in test_cases:

    result = analyze_symptoms(symptoms)

    print("=" * 60)
    print("Symptoms:", symptoms)
    print("Possible Category:", result["condition"])
    print("Care Level:", result["care_level"])
    print("Matched Symptoms:", result["matched_symptoms"])
    print("Explanation:", result["explanation"])

Symptoms: I have a cough, sore throat, and runny nose.
Possible Category: Common Cold
Care Level: Self-Care
Matched Symptoms: ['cough', 'runny nose', 'sore throat']
Explanation: These symptoms are commonly associated with mild respiratory conditions that can often be monitored at home.
Symptoms: I have a fever, chills, headache, and body aches.
Possible Category: Flu-Like Symptoms
Care Level: Doctor Visit
Matched Symptoms: ['fever', 'chills', 'body aches']
Explanation: Fever combined with body aches, fatigue, or chills may require medical evaluation, especially if symptoms become severe.
Symptoms: I have severe stomach pain and persistent vomiting.
Possible Category: Severe Abdominal Symptoms
Care Level: Urgent
Matched Symptoms: ['severe stomach pain', 'persistent vomiting']
Explanation: Severe abdominal symptoms may require prompt medical attention to determine their cause.
Symptoms: I have chest pain and difficulty breathing.
Possible Category: Breathing or Chest Emergency
Care Level

## 13. Create the Streamlit App

Now that the symptom analysis works, I can create a simple Streamlit interface. The interface lets the user enter symptoms and displays the category, care level, recognized symptoms, and explanation.

In [13]:
# Start a new Streamlit app file.

app_code = """
import streamlit as st
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))
"""

## 14. Add the Symptom Data

The same symptom categories tested earlier are added to the Streamlit application so the app can use them when analyzing user input.

In [14]:
app_code += """

# Create a small symptom knowledge base.
# Each category contains keywords, a care level, and an explanation.

symptom_database = {
    "General Headache Symptoms": {
        "keywords": [
            "headache",
            "mild headache"
        ],
        "care_level": "Self-Care",
        "explanation": (
            "A mild headache can often be monitored with rest, hydration, "
            "and other basic self-care unless it becomes severe or unusual."
        )
    },

    "Common Cold": {
        "keywords": [
            "cough",
            "runny nose",
            "stuffy nose",
            "sneezing",
            "sore throat"
        ],
        "care_level": "Self-Care",
        "explanation": (
            "These symptoms are commonly associated with mild respiratory "
            "conditions that can often be monitored at home."
        )
    },

    "Flu-Like Symptoms": {
        "keywords": [
            "fever",
            "chills",
            "body aches",
            "fatigue"
        ],
        "care_level": "Doctor Visit",
        "explanation": (
            "Fever combined with body aches, fatigue, or chills may require "
            "medical evaluation, especially if symptoms become severe."
        )
    },

    "Possible Infection": {
        "keywords": [
            "high fever",
            "swelling",
            "pus",
            "infection",
            "severe sore throat"
        ],
        "care_level": "Doctor Visit",
        "explanation": (
            "These symptoms may indicate an infection that could require "
            "evaluation and treatment by a healthcare professional."
        )
    },

    "Severe Abdominal Symptoms": {
        "keywords": [
            "severe stomach pain",
            "severe abdominal pain",
            "persistent vomiting",
            "blood in stool"
        ],
        "care_level": "Urgent",
        "explanation": (
            "Severe abdominal symptoms may require prompt medical attention "
            "to determine their cause."
        )
    },

    "Breathing or Chest Emergency": {
        "keywords": [
            "chest pain",
            "difficulty breathing",
            "shortness of breath",
            "cannot breathe",
            "blue lips"
        ],
        "care_level": "Emergency",
        "explanation": (
            "Chest pain or serious breathing problems can be signs of a "
            "medical emergency and should receive immediate attention."
        )
    },

    "Neurological Emergency": {
        "keywords": [
            "face drooping",
            "slurred speech",
            "sudden weakness",
            "seizure",
            "unconscious"
        ],
        "care_level": "Emergency",
        "explanation": (
            "Sudden neurological symptoms can indicate a serious emergency "
            "and require immediate professional medical care."
        )
    }
}
"""


## 15. Add the Processing Function

The same NLP function is used inside the app so both the user input and stored symptom keywords are processed in the same way.

In [15]:
app_code += """

def process_symptoms(text):
    # Convert the text to lowercase.
    text = text.lower()

    # Split the text into words.
    tokens = word_tokenize(text)

    # Remove punctuation and common stop words.
    filtered_tokens = [
        word for word in tokens
        if word.isalnum() and word not in stop_words
    ]

    # Reduce words to their basic form.
    processed_tokens = [
        lemmatizer.lemmatize(word)
        for word in filtered_tokens
    ]

    return processed_tokens
"""

## 16. Add the Symptom Analysis

This function compares the user's symptoms with the symptom database and selects the strongest match. Single-word symptoms and multi-word phrases are handled separately so inputs such as `I have a headache` can be recognized correctly. If two categories have the same score, the more serious care level receives priority.


In [16]:
app_code += """

# Assign a priority to each care level.
care_priority = {
    "Self-Care": 1,
    "Doctor Visit": 2,
    "Urgent": 3,
    "Emergency": 4
}


def analyze_symptoms(symptoms):
    # Compare the user's symptoms with the symptom knowledge base.
    processed_tokens = process_symptoms(symptoms)
    processed_text = " ".join(processed_tokens)

    best_match = None
    highest_score = 0
    highest_priority = 0

    for condition, information in symptom_database.items():
        current_matches = []

        for keyword in information["keywords"]:
            keyword_tokens = process_symptoms(keyword)

            # Match single-word symptoms directly in the processed word list.
            if len(keyword_tokens) == 1:
                if keyword_tokens[0] in processed_tokens:
                    current_matches.append(keyword)

            # Match longer symptom phrases in the processed text.
            else:
                processed_keyword = " ".join(keyword_tokens)
                if processed_keyword in processed_text:
                    current_matches.append(keyword)

        score = len(current_matches)
        priority = care_priority[information["care_level"]]

        if score > highest_score or (
            score == highest_score
            and score > 0
            and priority > highest_priority
        ):
            highest_score = score
            highest_priority = priority

            best_match = {
                "condition": condition,
                "care_level": information["care_level"],
                "explanation": information["explanation"],
                "matched_symptoms": current_matches,
                "score": score
            }

    if best_match is None:
        return {
            "condition": "No Clear Match",
            "care_level": "Doctor Visit",
            "explanation": (
                "The symptoms did not clearly match the categories in this "
                "project. Consider speaking with a healthcare professional "
                "if symptoms continue or become concerning."
            ),
            "matched_symptoms": [],
            "score": 0
        }

    return best_match
"""


## 17. Create the User Interface

The final part of the application creates the page title, symptom text box, analysis button, and result display.

In [17]:
app_code += """

st.set_page_config(
    page_title="AI Symptom & Care Guide",
    page_icon="🩺"
)

st.title("🩺 AI Symptom & Care Guide")

st.write(
    "Describe the symptoms you are experiencing, and the app will "
    "provide a general care recommendation."
)

user_symptoms = st.text_area(
    "Enter your symptoms:",
    placeholder="Example: I have a fever, headache, body aches, and chills."
)

if st.button("Analyze Symptoms"):

    if not user_symptoms.strip():
        st.warning("Please enter at least one symptom.")

    else:
        result = analyze_symptoms(user_symptoms)

        st.subheader("Analysis Result")

        st.write("**Possible Category:**", result["condition"])
        st.write("**Care Level:**", result["care_level"])

        if result["matched_symptoms"]:
            st.write(
                "**Recognized Symptoms:**",
                ", ".join(result["matched_symptoms"])
            )

        st.write("**Explanation:**")
        st.write(result["explanation"])

        if result["care_level"] == "Emergency":
            st.error(
                "These symptoms may require immediate medical attention."
            )

        elif result["care_level"] == "Urgent":
            st.warning(
                "These symptoms may require prompt medical attention."
            )

st.markdown("---")

st.caption(
    "Disclaimer: This application is for educational purposes only. "
    "It does not provide a medical diagnosis and does not replace "
    "professional medical advice."
)
"""

## 18. Save the Application File

The completed Streamlit code is saved as `app.py` so it can be run as a web application.

In [18]:
# Save all parts of the application into one Python file.

with open("app.py", "w") as file:
    file.write(app_code)

print("app.py created successfully.")

app.py created successfully.


## 19. Run the Streamlit App

Now that the `app.py` file is created, I can start the Streamlit server and open it from Google Colab. I use a temporary Cloudflare tunnel so the app can be viewed in a browser without needing an account or authentication token.

In [19]:
# Install Streamlit and download Cloudflared.

!pip install streamlit -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

print("Setup complete.")

Setup complete.


## 20. Start the App

This cell starts the Streamlit application in the background on port 8501.

In [20]:
# Stop an older Streamlit process if one is still running.
!pkill -f "streamlit run app.py" 2>/dev/null || true

# Start the app in the background.
!streamlit run app.py --server.port 8501 --server.headless true &>/content/streamlit_logs.txt &

import time
time.sleep(3)

print("Streamlit started.")

^C
Streamlit started.


## 21. Create a Public Link

The next cell creates a temporary public link for the Streamlit app. The link only works while this Colab session is running.

In [21]:
# Create a temporary Cloudflare link for the app.

import subprocess
import re
import time

# Close an older tunnel if this cell has already been run.
if "cloudflared_process" in globals():
    try:
        cloudflared_process.terminate()
    except:
        pass

cloudflared_process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None

for _ in range(60):
    line = cloudflared_process.stdout.readline()

    if line:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)

        if match:
            public_url = match.group(0)
            break

    time.sleep(0.5)

if public_url:
    print("Open the app here:")
    print(public_url)
else:
    print("The public link was not created. Run this cell again.")

Open the app here:
https://weeks-brakes-produce-pensions.trycloudflare.com


## 22. Test the Application

I tested the app with different symptom descriptions to make sure it could return different care levels.

Example inputs:

- `I have a cough, sore throat, and runny nose.`
- `I have fever, chills, headache, and body aches.`
- `I have severe stomach pain and persistent vomiting.`
- `I have chest pain and difficulty breathing.`

These examples test Self-Care, Doctor Visit, Urgent, and Emergency recommendations.

## 23. Testing the Application

I tested the application with different symptom descriptions to make sure the matching system and care levels worked as expected.

The tests included mild symptoms, flu-like symptoms, and symptoms that were not included in the knowledge base. This helped me check whether the application could return a matching category when possible and avoid guessing when no clear match was found.

In [22]:
test_cases = [
    "I have a headache",
    "I have fever, chills, headache, and body aches",
    "I have a cough, sore throat, and runny nose",
    "I have severe stomach pain and persistent vomiting",
    "I have chest pain and difficulty breathing",
    "I am feeling dizzy"
]

for symptoms in test_cases:
    result = analyze_symptoms(symptoms)

    print("Symptoms:", symptoms)
    print("Category:", result["condition"])
    print("Care Level:", result["care_level"])
    print("Matched Symptoms:", result["matched_symptoms"])
    print("-" * 50)

Symptoms: I have a headache
Category: General Headache Symptoms
Care Level: Self-Care
Matched Symptoms: ['headache']
--------------------------------------------------
Symptoms: I have fever, chills, headache, and body aches
Category: Flu-Like Symptoms
Care Level: Doctor Visit
Matched Symptoms: ['fever', 'chills', 'body aches']
--------------------------------------------------
Symptoms: I have a cough, sore throat, and runny nose
Category: Common Cold
Care Level: Self-Care
Matched Symptoms: ['cough', 'runny nose', 'sore throat']
--------------------------------------------------
Symptoms: I have severe stomach pain and persistent vomiting
Category: Severe Abdominal Symptoms
Care Level: Urgent
Matched Symptoms: ['severe stomach pain', 'persistent vomiting']
--------------------------------------------------
Symptoms: I have chest pain and difficulty breathing
Category: Breathing or Chest Emergency
Care Level: Emergency
Matched Symptoms: ['chest pain', 'difficulty breathing']
----------

## 24. Testing Results

The application correctly identified several symptom patterns from the knowledge base. A headache was classified as General Headache Symptoms, while fever, chills, and body aches were classified as Flu-Like Symptoms.

More serious symptom combinations, such as severe stomach pain or difficulty breathing, were assigned higher care levels. When I entered dizziness, the application returned No Clear Match because dizziness was not included in the current symptom database.

This test showed that the application only works with the symptoms and categories included in its knowledge base. A larger and more detailed dataset would improve the application in the future.

## 25. Conclusion

This project helped me understand how a simple AI healthcare application can use Natural Language Processing to analyze user-entered symptoms. I used tokenization, stop-word removal, lemmatization, and keyword matching to connect symptom descriptions with possible categories and care levels.

The application worked well with the symptom patterns included in the knowledge base and was able to separate mild, urgent, and emergency-level cases. I also learned that the results depend heavily on the quality and size of the symptom database. If I continued this project, I would add more symptoms, more categories, and a larger medical dataset to improve the accuracy of the recommendations.

Overall, this project gave me practical experience with NLP, Streamlit, testing, and building a simple AI-driven healthcare application.